# Phase 4 · Support Vector Machine classification

This notebook evaluates a Support Vector Machine with the same grouped holdout protocol used by `03_phase3_DT.ipynb`. Pixels from the same polygon never cross the train/test boundary. Hyper-parameters are selected by grouped cross-validation on the training partition only, using macro-F1 as the primary criterion.

The resulting tables and MLflow run use the same accuracy, kappa, balanced accuracy, precision, recall and macro-F1 measures as the Decision Tree notebook.

In [ ]:
import io
import json
import os
from pathlib import Path

import boto3
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from sqlalchemy import URL, create_engine, text
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, cohen_kappa_score,
                             confusion_matrix, f1_score, precision_recall_fscore_support)
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

DATASET = "T18NWL_20260810T152649"
BANDS = [band.strip() for band in os.environ["BAND_NAMES"].split(",")]
SEED = 42
N_SPLITS = 5
OUTPUTS = Path("outputs/phase4")
OUTPUTS.mkdir(parents=True, exist_ok=True)
print({name: os.environ.get(name, "<missing>") for name in ("MLFLOW_TRACKING_URI", "MLFLOW_S3_ENDPOINT_URL", "DATA_BUCKET", "TRAIN_DB_HOST")})

## 1. Load the training data and recover polygon groups

The TSV comes from the `extract_training_samples` DAG. The polygon identifier is joined from `extracted.polygon_pixels`, matching the data preparation used in phases 1–3.

In [ ]:
s3 = boto3.client("s3", endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"])
body = s3.get_object(Bucket=os.environ["DATA_BUCKET"], Key=f"tsv/{DATASET}.tsv")["Body"].read()
samples = pd.read_csv(io.BytesIO(body), sep="\t", float_precision="round_trip")
expected = ["Latitude", "Longitude", *BANDS, "Class"]
assert list(samples.columns) == expected, samples.columns.tolist()
assert not samples.isna().any().any()
engine = create_engine(URL.create("postgresql+psycopg2", username=os.environ["TRAIN_DB_USER"],
    password=os.environ["TRAIN_DB_PASSWORD"], host=os.environ["TRAIN_DB_HOST"],
    port=int(os.environ["TRAIN_DB_PORT"]), database=os.environ["TRAIN_DB_NAME"]))
with engine.connect() as connection:
    polygons = pd.read_sql(text('SELECT latitude AS "Latitude", longitude AS "Longitude", fid FROM extracted.polygon_pixels WHERE dataset = :dataset'), connection, params={"dataset": DATASET})
samples = samples.merge(polygons, on=["Latitude", "Longitude"], how="left", validate="one_to_one")
assert samples["fid"].notna().all(), "Some samples have no polygon group"
samples["fid"] = samples["fid"].astype(int)
X, y, groups = samples[BANDS], samples["Class"], samples["fid"]
print(f"Loaded {len(samples):,} samples and {groups.nunique():,} polygon groups")
samples.head()

## 2. Comparable grouped holdout

The first fold is held out exactly as in phase 3. All SVM tuning below sees only the training partition.

In [ ]:
splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
train_index, test_index = next(splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_index], X.iloc[test_index]
y_train, y_test = y.iloc[train_index], y.iloc[test_index]
g_train, g_test = groups.iloc[train_index], groups.iloc[test_index]
assert set(g_train).isdisjoint(set(g_test))
EVAL_LABELS = sorted(y_test.unique().tolist())
ALL_LABELS = sorted(y.unique().tolist())
split_summary = pd.DataFrame({
    "partition": ["train", "test"],
    "samples": [len(y_train), len(y_test)],
    "polygons": [g_train.nunique(), g_test.nunique()],
    "classes": [y_train.nunique(), y_test.nunique()],
})
split_summary.to_csv(OUTPUTS / "split.csv", index=False)
split_summary

## 3. SVM validation and tuning

Scaling is fitted inside every training fold. The grid is deliberately compact so each candidate is inspectable: linear and RBF kernels, three C values, and the RBF `scale` gamma.

In [ ]:
pipe = Pipeline([("scaler", StandardScaler()), ("svc", SVC())])
param_grid = {
    "svc__kernel": ["linear", "rbf"],
    "svc__C": [0.1, 1.0, 10.0],
    "svc__gamma": ["scale"],
}
cv = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(X_train, y_train, g_train))
cv_rows = []
for candidate_id, params in enumerate(ParameterGrid(param_grid), start=1):
    fold_metrics = []
    for fold, (fit_idx, val_idx) in enumerate(cv, start=1):
        model = clone(pipe).set_params(**params)
        model.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
        pred = model.predict(X_train.iloc[val_idx])
        fold_metrics.append({
            "accuracy": accuracy_score(y_train.iloc[val_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y_train.iloc[val_idx], pred),
            "kappa": cohen_kappa_score(y_train.iloc[val_idx], pred),
            "macro_f1": f1_score(y_train.iloc[val_idx], pred, average="macro", zero_division=0),
        })
    row = {**params, "candidate_id": candidate_id}
    for metric in ("accuracy", "balanced_accuracy", "kappa", "macro_f1"):
        values = [item[metric] for item in fold_metrics]
        row[f"cv_{metric}_mean"] = float(np.mean(values))
        row[f"cv_{metric}_std"] = float(np.std(values, ddof=1))
    cv_rows.append(row)
cv_results = pd.DataFrame(cv_rows).sort_values("cv_macro_f1_mean", ascending=False).reset_index(drop=True)
cv_results.to_csv(OUTPUTS / "cv_results.csv", index=False)
best_params = {key: value for key, value in cv_results.iloc[0].to_dict().items() if key in param_grid}
best_params, cv_results

## 4. Holdout evaluation and MLflow

The selected configuration is refit on all training pixels and evaluated once on the untouched polygon holdout.

In [ ]:
def evaluate(y_true, predicted):
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, predicted, labels=EVAL_LABELS, average=None, zero_division=0)
    macro = precision_recall_fscore_support(y_true, predicted, labels=EVAL_LABELS, average="macro", zero_division=0)
    metrics = {"overall_accuracy": float(accuracy_score(y_true, predicted)), "kappa": float(cohen_kappa_score(y_true, predicted)),
               "balanced_accuracy": float(balanced_accuracy_score(y_true, predicted)), "precision_macro": float(macro[0]),
               "recall_macro": float(macro[1]), "f1_macro": float(macro[2])}
    per_class = pd.DataFrame({"precision": precision, "recall": recall, "f1": f1}, index=EVAL_LABELS)
    return metrics, per_class

selected = clone(pipe).set_params(**best_params)
selected.fit(X_train, y_train)
predicted = selected.predict(X_test)
metrics, per_class = evaluate(y_test, predicted)
per_class.index.name = "Class"
per_class.to_csv(OUTPUTS / "per_class_svm.csv")
matrix = pd.DataFrame(confusion_matrix(y_test, predicted, labels=ALL_LABELS), index=ALL_LABELS, columns=ALL_LABELS)
matrix.index.name, matrix.columns.name = "reference", "predicted"
matrix.to_csv(OUTPUTS / "confusion_matrix_svm.csv")
fig, ax = plt.subplots(figsize=(7, 6))
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(y_test, predicted, labels=EVAL_LABELS, normalize="true", values_format=".2f", cmap="Blues", colorbar=False, ax=ax)
ax.set(title=f"SVM · OA {metrics['overall_accuracy']:.3f} · macro-F1 {metrics['f1_macro']:.3f}", xlabel="Predicted class", ylabel="Reference class")
fig.tight_layout()
fig.savefig(OUTPUTS / "confusion_matrix_svm.png", dpi=150)
plt.show()

mlflow.set_experiment(os.environ["MLFLOW_EXPERIMENT_NAME"])
with mlflow.start_run(run_name="phase4-svm") as run:
    mlflow.log_params({"phase": 4, "model": "Support Vector Machine", "dataset": DATASET, "bands": ",".join(BANDS), "seed": SEED, "validation": f"StratifiedGroupKFold {N_SPLITS} folds by polygon", **{f"svc_{k}": v for k, v in best_params.items()}})
    mlflow.log_metrics({**metrics, "cv_macro_f1_mean": float(cv_results.iloc[0]["cv_macro_f1_mean"])})
    for artifact in ("cv_results.csv", "split.csv", "per_class_svm.csv", "confusion_matrix_svm.csv", "confusion_matrix_svm.png"):
        mlflow.log_artifact(OUTPUTS / artifact)
    mlflow.sklearn.log_model(selected, name="model")
    run_id = run.info.run_id

summary = pd.DataFrame([{"model": "SVM", **metrics, "cv_macro_f1_mean": float(cv_results.iloc[0]["cv_macro_f1_mean"]), "mlflow_run_id": run_id}])
summary.to_csv(OUTPUTS / "summary.csv", index=False)
(OUTPUTS / "metrics.json").write_text(json.dumps({"dataset": DATASET, "train_samples": int(len(y_train)), "test_samples": int(len(y_test)), "evaluated_classes": [int(v) for v in EVAL_LABELS], "best_params": best_params, "cv_macro_f1_mean": float(cv_results.iloc[0]["cv_macro_f1_mean"]), "holdout_metrics": {key: float(value) for key, value in metrics.items()}, "mlflow_run_id": run_id}, indent=2))
summary.round(4)

## 5. Comparison with phase 3

Load the Decision Tree summary if it exists. The columns are intentionally aligned so the two approaches can be compared on the same holdout and metrics.

In [ ]:
phase3_path = Path("outputs/phase3/summary.csv")
if phase3_path.exists():
    phase3 = pd.read_csv(phase3_path)
    phase3["model"] = phase3["model"].str.replace("Decision Tree · ", "Decision Tree · ", regex=False)
    comparison = pd.concat([phase3[["model", "overall_accuracy", "kappa", "balanced_accuracy", "precision_macro", "recall_macro", "f1_macro"]], summary.drop(columns="mlflow_run_id")], ignore_index=True)
else:
    comparison = summary.drop(columns="mlflow_run_id")
comparison.to_csv(OUTPUTS / "phase3_vs_phase4.csv", index=False)
comparison.round(4)